In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
torch.set_default_dtype(torch.float32)

from src.cann.models.minimal import Minimal
from src.cann.models.polykan import PolyKAN
from src.cann.models.base import ActNN
from src.cann.train import ACT_DICT

# Minimal ReLU NN from Springer and Kenyon
minimal = Minimal()

In [ ]:
# make input grid, covering B/S neighborhoods from 0 through 8
n_inputs = torch.ones(200,1,3,3)

halfway = 100
dn = 9 / (halfway-1)
n_inputs[:halfway] *= torch.arange(0, 9+dn, dn).reshape(-1,1,1,1)

#for coef, xx in enumerate(range(0,90,10)):
#    n_inputs[xx:xx+10] *= (coef) 

n_inputs[halfway:] = n_inputs[:halfway]
n_inputs /= 8 
n_inputs[:halfway,:,1,1] *= 0.0

n_inputs[halfway:,:,1,1] = 1.0

In [ ]:

my_cmap = plt.get_cmap("plasma")

color_index = 32

results_dir = os.path.join("..","results")
list_dir = ["full_relu_l1_2",\
            "full_prelu_l1_1",\
            "full_poly_l1_1"]
    
    
for elem in zip(list_dir):

    df = pd.read_csv(os.path.join(results_dir, elem[0], "exp.csv"))
    if (df["final_grid_accuracy"] == 1.0).sum():
        print((df["final_grid_accuracy"] == 1.0).sum())
        entry = df.loc[df[df["final_grid_accuracy"] == 1.0].index[0]] 
        width = entry["model_width"]
        depth = entry["model_depth"]
                        
        run_df = pd.read_csv(os.path.join("..", entry["run_filename"]))
        params = np.load(os.path.join("..", run_df["parameters_filename"][0]))[-1]
        
        print(entry["run_filename"], entry["activation_name"], entry["model_width"])
        
        if entry["model_name"] == "PolyKAN":
            print("polykan")
            model = PolyKAN(width=width, depth=depth)
        else:
            activation = ACT_DICT[entry["activation_name"].lower()]
            model = ActNN(width=width, depth=depth, activation=activation,\
                         )

        model.set_parameters(params)
    
        with torch.no_grad():
            bs_out = model(n_inputs)[:,:,1,1]
        
        fig, ax = plt.subplots(1,1) #figure()
        ax.plot(bs_out[:halfway], "-", alpha=0.5, lw=3, color=my_cmap(color_index), label="B")
        ax.plot(bs_out[halfway:], "-", alpha=0.5, lw=3, color=my_cmap(255-color_index),label="S")
        
        ax.plot(bs_out[:halfway].round(), "--", alpha=0.25, lw=3, color=my_cmap(color_index), label="B(round)")
        ax.plot(bs_out[halfway:].round(), "--", alpha=0.25, lw=3, color=my_cmap(255-color_index),label="S(rounded)")
        
        moores = n_inputs[:halfway].sum((1,2,3))
        x_ticks = np.arange(0,halfway,10)
        x_ticklabels = moores[::10]
        
        for num, ss in enumerate(entry["rulestring"].split("/")[1][1:]):
            if num:
                ax.scatter(torch.where(n_inputs.sum((1,2,3)) == (float(ss)))[0][0], 1,  s=300,\
                           marker="s", alpha=0.1, edgecolors="k", lw=2, color=my_cmap(color_index-50))
            else:
                ax.scatter(torch.where(n_inputs.sum((1,2,3)) == (float(ss)))[0][0], 1,  s=300, \
                           marker="s", alpha=0.1, edgecolors="k", lw=2, color=my_cmap(color_index-50), label="S (ground truth)")
        for num, bb in enumerate(entry["rulestring"].split("/")[0][1:]):
            if num:
                ax.scatter(torch.where(n_inputs.sum((1,2,3)) == (float(bb)))[0][0], 1, s=300,\
                            alpha=0.3, color=my_cmap(color_index+50))
            else:
                ax.scatter(torch.where(n_inputs.sum((1,2,3)) == (float(bb)))[0][0], 1, s=300,\
                            alpha=0.3, color=my_cmap(color_index+50), label="B (ground truth)")

        ax.set_xticks([elem+5 for elem in x_ticks])
        ax.set_xticklabels([f"{elem.item():.2f}" for elem in x_ticklabels], rotation=45)
        
        plt.title(f"B/S equivalent for {entry['model_name']} {elem[0]}")
        plt.legend()
        plt.savefig(f"rule_viz{elem[0]}.png")
        plt.show()

    
        color_index += 32
            